<a href="https://colab.research.google.com/github/Marfall/GradientBoosting_Otus1/blob/main/GradientBoosting_Otus_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее Задание номер 1:     "Градиентный бустинг:"

## Импорт библиотек и настройка окружения

Подключаем все необходимые библиотеки:
- pandas, numpy для данных
- matplotlib, seaborn для графиков
- sklearn для предобработки, моделей и метрик
- xgboost, catboost, lightgbm — три специализированных бустинга

Отключаем предупреждения и настраиваем pandas на показ всех колонок.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

import xgboost as xgb
import catboost as cb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print("Библиотеки загружены.")

## Загрузка данных и первичный EDA

Пытаемся загрузить датасет Telco Churn:
- сначала с пути Kaggle
- если не найден — с локального файла.

Выводим:
- первые 5 строк
- информацию о колонках (типы, пропуски)
- статистику по числовым признакам
- распределение целевой переменной `Churn`
- столбчатую диаграмму для визуализации баланса классов

In [ ]:
try:
    df = pd.read_csv("/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    print("Данные загружены с Kaggle.")
except FileNotFoundError:
    df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
    print("Данные загружены локально.")

print("\nПервые 5 строк:")
display(df.head())

print("\nИнформация о датасете:")
df.info()

print("\nСтатистика по числовым признакам:")
display(df.describe())

print("\nПропуски по колонкам:")
print(df.isnull().sum())

print("\nРаспределение Churn:")
print(df['Churn'].value_counts())
print(f"Доля оттока: {df['Churn'].value_counts(normalize=True)['Yes']:.2%}")

plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Churn')
plt.title('Распределение целевой переменной (Churn)')
plt.show()

## Визуализация: распределения, ящики с усами и корреляции

Чтобы глубже понять данные, строим:
- **Гистограммы** распределения каждого числового признака.
- **Ящики с усами (boxplot)** для числовых признаков в разрезе целевой переменной (Churn) — это покажет, как распределены значения среди ушедших и оставшихся клиентов.
- **Матрицу корреляции** между числовыми признаками (heatmap) — выявляем линейные взаимосвязи.

In [ ]:
# Приводим TotalCharges к числу, чтобы работать с ним как с числовым
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)

# Числовые признаки (исключаем customerID и целевую)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [col for col in num_cols if col not in ['customerID', 'Churn']]

# 1. Гистограммы распределений
df[num_cols].hist(figsize=(12, 8), bins=30, edgecolor='black')
plt.suptitle('Распределение числовых признаков', size=16)
plt.tight_layout()
plt.show()

# 2. Boxplot по классу Churn для каждого числового признака
for col in num_cols:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x='Churn', y=col)
    plt.title(f'Boxplot: {col} vs Churn')
    plt.show()

# 3. Матрица корреляции (тепловая карта)
plt.figure(figsize=(10,8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Матрица корреляции числовых признаков')
plt.show()

## Предобработка через Pipeline

1. Приводим `TotalCharges` к числовому типу (уже сделано).
2. Определяем списки числовых и категориальных признаков (исключаем `customerID` и `Churn`).
3. Создаём трансформеры:
   - для числовых: `StandardScaler` (масштабирование)
   - для категориальных: `OrdinalEncoder` (превращает строки в числа)
4. Объединяем в `ColumnTransformer` — он применит разные преобразования к разным колонкам.
5. Отделяем признаки `X` и целевую `y` (Yes→1, No→0).
6. Разбиваем на train/test (80/20) со стратификацией по классам.

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols.remove('customerID')
categorical_cols.remove('Churn')

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Числовые:", numerical_cols)
print("Категориальные:", categorical_cols)

numeric_transformer = Pipeline([('scaler', StandardScaler())])
categorical_transformer = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} записей")
print(f"Тестовая выборка: {X_test.shape[0]} записей")

## Обучение моделей "из коробки"

Определяем функцию `evaluate_model`, которая:
- делает предсказания на тесте
- вычисляет Accuracy и ROC‑AUC
- выводит матрицу ошибок и отчёт по классификации

Обучаем 4 модели с параметрами по умолчанию:
1. `GradientBoostingClassifier` (sklearn)
2. `XGBoost`
3. `CatBoost`
4. `LightGBM`

Каждая модель обёрнута в Pipeline, включающий предобработку.

Результаты сохраняем в таблицу и визуализируем на графиках.

In [ ]:
def evaluate_model(model_pipeline, X_test, y_test, model_name):
    y_pred = model_pipeline.predict(X_test)
    y_proba = model_pipeline.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)

    print(f"\n=== {model_name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC-AUC:  {roc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))

    return {'model': model_name, 'accuracy': acc, 'roc_auc': roc}

results_default = []

pipeline_sklearn = Pipeline([('preprocessor', preprocessor),
                             ('classifier', GradientBoostingClassifier(random_state=42))])
pipeline_sklearn.fit(X_train, y_train)
results_default.append(evaluate_model(pipeline_sklearn, X_test, y_test, "sklearn GBDT (default)"))

pipeline_xgb = Pipeline([('preprocessor', preprocessor),
                         ('classifier', xgb.XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False))])
pipeline_xgb.fit(X_train, y_train)
results_default.append(evaluate_model(pipeline_xgb, X_test, y_test, "XGBoost (default)"))

pipeline_cat = Pipeline([('preprocessor', preprocessor),
                         ('classifier', cb.CatBoostClassifier(random_seed=42, verbose=False))])
pipeline_cat.fit(X_train, y_train)
results_default.append(evaluate_model(pipeline_cat, X_test, y_test, "CatBoost (default)"))

pipeline_lgb = Pipeline([('preprocessor', preprocessor),
                         ('classifier', lgb.LGBMClassifier(random_state=42, verbose=-1))])
pipeline_lgb.fit(X_train, y_train)
results_default.append(evaluate_model(pipeline_lgb, X_test, y_test, "LightGBM (default)"))

df_default = pd.DataFrame(results_default)
print("\nСравнение без настройки:")
print(df_default[['model', 'accuracy', 'roc_auc']])

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
sns.barplot(data=df_default, x='model', y='accuracy')
plt.title('Accuracy (default)')
plt.ylim(0.7, 0.85)
plt.xticks(rotation=45)

plt.subplot(1,2,2)
sns.barplot(data=df_default, x='model', y='roc_auc')
plt.title('ROC-AUC (default)')
plt.ylim(0.7, 0.9)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Настройка гиперпараметров (GridSearchCV)

Создаём функцию `tune_model`, которая:
- выполняет 5‑кратную кросс‑валидацию по ROC‑AUC
- перебирает заданную сетку параметров
- возвращает лучшую модель и её метрики на тесте

Для каждой модели задаём небольшой набор гиперпараметров:
- sklearn GBDT: `n_estimators`, `learning_rate`, `max_depth`
- XGBoost: дополнительно `subsample`
- CatBoost: `iterations`, `learning_rate`, `depth`
- LightGBM: `n_estimators`, `learning_rate`, `num_leaves`, `subsample`

После подбора выводим лучшие параметры и метрики, строим сравнительные графики.

In [ ]:
def tune_model(pipeline, param_grid, X_train, y_train, X_test, y_test, model_name):
    print(f"\nНастройка {model_name}...")
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0)
    grid.fit(X_train, y_train)

    best = grid.best_estimator_
    print(f"Лучшие параметры: {grid.best_params_}")

    y_pred = best.predict(X_test)
    y_proba = best.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    print(f"{model_name} tuned - Accuracy: {acc:.4f}, ROC-AUC: {roc:.4f}")

    return {'model': model_name + ' (tuned)', 'accuracy': acc, 'roc_auc': roc, 'best_params': grid.best_params_}

results_tuned = []

res_tuned = tune_model(
    Pipeline([('preprocessor', preprocessor),
              ('classifier', GradientBoostingClassifier(random_state=42))]),
    {'classifier__n_estimators': [50, 100],
     'classifier__learning_rate': [0.05, 0.1],
     'classifier__max_depth': [3, 4]},
    X_train, y_train, X_test, y_test, "sklearn GBDT"
)
results_tuned.append(res_tuned)

res_tuned = tune_model(
    Pipeline([('preprocessor', preprocessor),
              ('classifier', xgb.XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False))]),
    {'classifier__n_estimators': [50, 100],
     'classifier__learning_rate': [0.05, 0.1],
     'classifier__max_depth': [3, 4],
     'classifier__subsample': [0.8, 1.0]},
    X_train, y_train, X_test, y_test, "XGBoost"
)
results_tuned.append(res_tuned)

res_tuned = tune_model(
    Pipeline([('preprocessor', preprocessor),
              ('classifier', cb.CatBoostClassifier(random_seed=42, verbose=False))]),
    {'classifier__iterations': [100, 200],
     'classifier__learning_rate': [0.05, 0.1],
     'classifier__depth': [4, 6]},
    X_train, y_train, X_test, y_test, "CatBoost"
)
results_tuned.append(res_tuned)

res_tuned = tune_model(
    Pipeline([('preprocessor', preprocessor),
              ('classifier', lgb.LGBMClassifier(random_state=42, verbose=-1))]),
    {'classifier__n_estimators': [50, 100],
     'classifier__learning_rate': [0.05, 0.1],
     'classifier__num_leaves': [31, 50],
     'classifier__subsample': [0.8, 1.0]},
    X_train, y_train, X_test, y_test, "LightGBM"
)
results_tuned.append(res_tuned)

df_tuned = pd.DataFrame(results_tuned)
print("\nСравнение после настройки:")
print(df_tuned[['model', 'accuracy', 'roc_auc']])

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
sns.barplot(data=df_tuned, x='model', y='accuracy')
plt.title('Accuracy (tuned)')
plt.ylim(0.7, 0.85)
plt.xticks(rotation=45)

plt.subplot(1,2,2)
sns.barplot(data=df_tuned, x='model', y='roc_auc')
plt.title('ROC-AUC (tuned)')
plt.ylim(0.7, 0.9)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Итоговое сравнение и определение победителя

Объединяем результаты до и после настройки в одну таблицу.
Строим групповые столбчатые диаграммы для Accuracy и ROC‑AUC, где по оси X — модель, а цвет показывает тип (default/tuned).

Определяем лучшую модель:
- без настройки (по ROC‑AUC)
- после настройки (по ROC‑AUC)

Выводим итоговые названия победителей.

In [ ]:
df_default['type'] = 'default'
df_tuned_clean = df_tuned.copy()
df_tuned_clean['model'] = df_tuned_clean['model'].str.replace(' (tuned)', '')
df_tuned_clean['type'] = 'tuned'

df_combined = pd.concat([df_default, df_tuned_clean], ignore_index=True)
print("\nСравнительная таблица (default vs tuned):")
display(df_combined[['model', 'type', 'accuracy', 'roc_auc']])

fig, axes = plt.subplots(1, 2, figsize=(12,5))
sns.barplot(data=df_combined, x='model', y='accuracy', hue='type', ax=axes[0])
axes[0].set_title('Accuracy')
axes[0].legend(loc='lower right')

sns.barplot(data=df_combined, x='model', y='roc_auc', hue='type', ax=axes[1])
axes[1].set_title('ROC-AUC')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

best_default = df_default.loc[df_default['roc_auc'].idxmax()]
best_tuned = df_tuned.loc[df_tuned['roc_auc'].idxmax()]

print("\n=== ПОБЕДИТЕЛИ ===")
print(f"Без настройки: {best_default['model']} (ROC-AUC = {best_default['roc_auc']:.4f})")
print(f"После настройки: {best_tuned['model']} (ROC-AUC = {best_tuned['roc_auc']:.4f})")

## Важность признаков (Feature Importance)

Извлекаем важность признаков из лучшей настроенной модели (той, что дала максимальный ROC‑AUC).
Это позволяет понять, какие факторы сильнее всего влияют на отток клиентов.

Для XGBoost / LightGBM / sklearn GBDT важность считается как среднее уменьшение примесей (или gain).
Для CatBoost — аналогично.

Строим горизонтальную столбчатую диаграмму, отсортированную по убыванию важности.
Выводим также список признаков с их вкладом.

In [ ]:
# Определяем лучшую модель по ROC‑AUC после настройки
best_model_name = df_tuned.loc[df_tuned['roc_auc'].idxmax(), 'model']
best_model_roc = df_tuned.loc[df_tuned['roc_auc'].idxmax(), 'roc_auc']

print(f"Лучшая модель: {best_model_name} (ROC-AUC = {best_model_roc:.4f})")

# Извлекаем обученный классификатор из пайплайна
# Для этого нужно найти соответствующую обученную модель среди настроенных
# Мы сохранили лучшие модели в процессе настройки, но не сохранили их в переменные.
# Поэтому пересоздадим лучшую модель отдельно для визуализации важности.

# Определяем, какая модель победила, и создаём её заново с лучшими параметрами
if 'CatBoost' in best_model_name:
    best_params = df_tuned.loc[df_tuned['model'] == best_model_name, 'best_params'].values[0]
    best_clf = cb.CatBoostClassifier(random_seed=42, verbose=False, **best_params)
elif 'XGBoost' in best_model_name:
    best_params = df_tuned.loc[df_tuned['model'] == best_model_name, 'best_params'].values[0]
    best_clf = xgb.XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False, **best_params)
elif 'LightGBM' in best_model_name:
    best_params = df_tuned.loc[df_tuned['model'] == best_model_name, 'best_params'].values[0]
    best_clf = lgb.LGBMClassifier(random_state=42, verbose=-1, **best_params)
else:  # sklearn GBDT
    best_params = df_tuned.loc[df_tuned['model'] == best_model_name, 'best_params'].values[0]
    best_clf = GradientBoostingClassifier(random_state=42, **best_params)

# Обучаем финальный пайплайн с лучшим классификатором
best_pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', best_clf)])
best_pipeline.fit(X_train, y_train)

# Извлекаем важность признаков
if 'CatBoost' in best_model_name:
    importances = best_clf.feature_importances_
elif 'XGBoost' in best_model_name:
    importances = best_clf.feature_importances_
elif 'LightGBM' in best_model_name:
    importances = best_clf.feature_importances_
else:  # sklearn GBDT
    importances = best_clf.feature_importances_

# Получаем имена признаков после предобработки (в том же порядке, что в пайплайне)
# Для ColumnTransformer можно получить имена колонок
feature_names = []
for name, transformer, columns in preprocessor.transformers_:
    if name == 'num':
        feature_names.extend(columns)
    elif name == 'cat':
        feature_names.extend(columns)

# Создаём DataFrame с важностью
fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False)

print("\nТоп-10 наиболее важных признаков:")
display(fi_df.head(10))

# График важности
plt.figure(figsize=(10,6))
sns.barplot(data=fi_df.head(15), x='importance', y='feature')
plt.title(f'Важность признаков (лучшая модель: {best_model_name})')
plt.xlabel('Важность')
plt.tight_layout()
plt.show()